# Hierarchical Weakest-Link Classification (Upper/Lower → Specific)

Goal:
- Baseline: one model predicts the weakest-link class (all classes)
- Hierarchical: Stage 1 predicts Upper vs Lower, Stage 2 predicts the specific class within that region

Also supports:
- Option to drop selected weakest-link classes (for comparison)
- Additional metric: R² (computed from one-hot labels vs predicted probabilities)


## Imports

In [172]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    log_loss, r2_score, classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## Ignore warnings

In [173]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)


## Configs + toggles

In [174]:
RANDOM_SEED = 42

# Toggle: run with all classes or drop some (for A/B comparison)
DROP_CLASSES = True
DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift",
}

# How to handle ties (multiple indicators share max)
DROP_TIES = True

# Update paths if needed (relative to your notebook location)
SCORES_PATH = "../../data/processed/20190108_scores_and_weak_links.csv"
FEATURES_PATH = "../../data/processed/AimoScore_WeakLink_big_scores.csv"


## Load the CVS

In [175]:
scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

print("scores:", scores.shape)
print("big:", big.shape)
display(scores.head(2))
display(big.head(2))


scores: (2096, 17)
big: (2094, 43)


,ID,Date,SCORE,ForwardHead,LeftArmFallForward,RightArmFallForward,LeftShoulderElevation,RightShoulderElevation,ExcessiveForwardLean,LeftAsymmetricalWeightShift,RightAsymmetricalWeightShift,LeftKneeMovesInward,RightKneeMovesInward,LeftKneeMovesOutward,RightKneeMovesOutward,LeftHeelRises,RightHeelRises
0,cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect,2018-07-19,0.612532,0.31,1.34,1.15,0.33,0.49,0.00,0.0,0.08,0.0,0.0,0.0,0.77,0.0,0.0
1,6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect,2018-07-30,0.388809,1.49,1.42,1.39,0.00,0.00,1.08,0.1,0.00,0.0,0.0,0.0,0.40,0.0,0.0


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


## Build labels (dropping preformed here)

In [176]:
NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

if DROP_CLASSES:
    indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

# Ensure numeric
scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

if DROP_TIES:
    scores_labeled = scores.loc[~scores["HasTie"]].copy()
else:
    scores_labeled = scores.copy()

scores_labeled = scores_labeled.dropna(subset=["WeakestLink"])

print("Indicator classes used:", len(indicator_cols))
print("Tie rows:", int(scores["HasTie"].sum()))
print("Rows kept:", scores_labeled.shape[0])
print("Unique classes:", scores_labeled["WeakestLink"].nunique())
display(scores_labeled["WeakestLink"].value_counts())


Indicator classes used: 10
Tie rows: 372
Rows kept: 1724
Unique classes: 10


WeakestLink
RightArmFallForward            454
LeftArmFallForward             327
RightKneeMovesOutward          279
RightShoulderElevation         252
ExcessiveForwardLean            99
ForwardHead                     96
LeftAsymmetricalWeightShift     71
LeftShoulderElevation           51
LeftKneeMovesOutward            49
RightKneeMovesInward            46
Name: count, dtype: int64

In [177]:
labels = scores_labeled[["ID", "WeakestLink"]]
data = big.merge(labels, on="ID", how="inner")

print("After merge:", data.shape)
print("Unique classes:", data["WeakestLink"].nunique())

angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

print("Angle features:", len(angle_cols))
print("NASM features:", len(nasm_cols))
print("Total features:", len(feature_cols))

# --- Build feature matrix X and label vector y ---
X = data[feature_cols].copy()
y = data["WeakestLink"].astype(str).copy()

# Ensure numeric + drop rows with any NaNs in X
X = X.apply(pd.to_numeric, errors="coerce")
mask = X.notna().all(axis=1)
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("After dropping NaNs:", X.shape, y.shape)
print("Min class count:", y.value_counts().min())



After merge: (1724, 44)
Unique classes: 10
Angle features: 13
NASM features: 25
Total features: 38
After dropping NaNs: (1724, 38) (1724,)
Min class count: 46


## Define upper vs lower mapping 

In [178]:
UPPER = {
    "ForwardHead",
    "LeftArmFallForward",
    "RightArmFallForward",
    "LeftShoulderElevation",
    "RightShoulderElevation",
}
TRUNK = {"ExcessiveForwardLean"}

def to_region2(lbl: str) -> str:
    return "Upper" if lbl in UPPER else "Lower"

def to_region3(lbl: str) -> str:
    if lbl in UPPER: return "Upper"
    if lbl in TRUNK: return "Trunk"
    return "Lower"

region  = y.map(to_region2)
region3 = y.map(to_region3)

print("\n2-way region counts:")
print(region.value_counts())

print("\n3-way region counts:")
print(region3.value_counts())


2-way region counts:
WeakestLink
Upper    1181
Lower     543
Name: count, dtype: int64

3-way region counts:
WeakestLink
Upper    1181
Lower     444
Trunk      99
Name: count, dtype: int64


## Used metrics

In [179]:


def align_proba_to_all_classes(proba, model_classes, all_classes):
    """
    Expand/reorder proba columns so they match all_classes.
    Missing classes get probability 0.
    """
    model_classes = list(model_classes)
    all_classes = list(all_classes)
    out = np.zeros((proba.shape[0], len(all_classes)), dtype=float)
    idx = {c: i for i, c in enumerate(all_classes)}
    for j, c in enumerate(model_classes):
        out[:, idx[c]] = proba[:, j]
    return out

def multiclass_r2_from_proba(y_true, proba_aligned, classes):
    """
    R² between one-hot(y_true) and predicted probabilities (aligned).
    """
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([cls_to_idx[t] for t in y_true])
    y_onehot = np.zeros((len(y_true), len(classes)), dtype=float)
    y_onehot[np.arange(len(y_true)), y_idx] = 1.0
    return r2_score(y_onehot, proba_aligned, multioutput="uniform_average")

def summarize_metrics(y_true, y_pred, proba, all_classes):
    """
    proba must already be aligned to all_classes.
    """
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "log_loss": log_loss(y_true, proba, labels=list(all_classes)),
        "r2_onehot_vs_proba": multiclass_r2_from_proba(y_true, proba, list(all_classes)),
    }


## Model builder

In [180]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

baseline_models = {
    "logreg_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),

    "lightgbm_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
}


In [181]:
def _is_xgb_pipeline(est):
    # True if final estimator is an XGBClassifier (whether pipeline or not)
    if isinstance(est, Pipeline):
        return isinstance(est.steps[-1][1], XGBClassifier)
    return isinstance(est, XGBClassifier)
    
def evaluate_model_cv(model, X, y, skf, all_classes):
    """
    Cross-validated evaluation for multiclass models.

    Fix: XGBoost expects numeric y in many sklearn-wrapper versions.
    We label-encode y *only for XGB* during training/prediction.
    """
    rows = []
    all_classes = list(all_classes)

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        m = clone(model)

        # --- Special handling for XGBClassifier: encode y to 0..K-1 ---
        if _is_xgb_pipeline(m):
            le = LabelEncoder()
            le.fit(all_classes)  # ensure stable mapping across folds
            ytr_enc = le.transform(ytr)
            yte_enc = le.transform(yte)

            m.fit(Xtr, ytr_enc)

            proba_raw = m.predict_proba(Xte)
            ypred_enc = m.predict(Xte)
            ypred = pd.Series(le.inverse_transform(ypred_enc), index=yte.index)

            # For metrics, we need y_true as strings + proba aligned to string classes
            # XGB proba columns correspond to encoded class order = le.classes_
            proba = align_proba_to_all_classes(proba_raw, le.classes_, all_classes)

            metrics = summarize_metrics(yte, ypred, proba, all_classes)

        else:
            # Normal sklearn classifiers can handle string labels
            m.fit(Xtr, ytr)

            proba_raw = m.predict_proba(Xte)
            if hasattr(m, "classes_"):
                model_classes = m.classes_
            else:
                model_classes = m.steps[-1][1].classes_

            proba = align_proba_to_all_classes(proba_raw, model_classes, all_classes)
            ypred = m.predict(Xte)

            metrics = summarize_metrics(yte, ypred, proba, all_classes)

        metrics["fold"] = fold
        rows.append(metrics)

    return pd.DataFrame(rows)


## Adjust splits

In [182]:
min_class_count = int(y.value_counts().min())
n_splits = min(5, min_class_count)

if n_splits < 2:
    raise ValueError(f"Not enough samples for CV. Min class count = {min_class_count}")

print("Min class count:", min_class_count)
print("Using n_splits:", n_splits)

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

Min class count: 46
Using n_splits: 5


## Baseline CV evaluation

In [183]:
all_classes = sorted(y.unique())

baseline_results = []
baseline_details = {}  # store per-fold dfs

for name, model in baseline_models.items():
    df = evaluate_model_cv(model, X, y, skf, all_classes)
    baseline_details[name] = df

    mean_row = df.mean(numeric_only=True).to_dict()
    mean_row["model"] = name
    baseline_results.append(mean_row)

baseline_summary = (
    pd.DataFrame(baseline_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(baseline_summary)


,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,r2_onehot_vs_proba,fold,model
0,0.658342,0.542838,0.555771,0.650334,1.623420,0.269642,3.0,lightgbm_balanced
1,0.541164,0.552853,0.484739,0.553784,1.273732,0.147999,3.0,logreg_balanced


In [184]:
# Stage 1 candidates (binary Upper/Lower)
stage1_models = {
    "s1_logreg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),
    "s1_lightgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
}

# Stage 2 candidates (multiclass within region)
stage2_upper_models = {
    "upper_lgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
}

stage2_lower_models = {
    "lower_lgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=800,        # often helps lower region (more variety)
            learning_rate=0.03,
            num_leaves=63,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
}



## Evaluate stage 1 models

In [185]:
# -------------------------
# CV helper (baseline multiclass)
# -------------------------
def evaluate_model_cv(model, X, y, skf, all_classes):
    rows = []
    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        m = clone(model)
        m.fit(Xtr, ytr)

        proba_raw = m.predict_proba(Xte)
        if hasattr(m, "classes_"):
            model_classes = m.classes_
        else:
            model_classes = m.steps[-1][1].classes_

        proba = align_proba_to_all_classes(proba_raw, model_classes, all_classes)
        ypred = m.predict(Xte)

        metrics = summarize_metrics(yte, ypred, proba, all_classes)
        metrics["fold"] = fold
        rows.append(metrics)

    return pd.DataFrame(rows)

# -------------------------
# Stage 1 (region) CV helper
# -------------------------
def evaluate_stage1_cv(stage1, X, region_labels, skf):
    rows = []
    for fold, (tr, te) in enumerate(skf.split(X, region_labels), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        rtr, rte = region_labels.iloc[tr], region_labels.iloc[te]

        m = clone(stage1)
        m.fit(Xtr, rtr)
        pred = m.predict(Xte)

        rows.append({
            "fold": fold,
            "accuracy": accuracy_score(rte, pred),
            "balanced_accuracy": balanced_accuracy_score(rte, pred),
            "macro_f1": f1_score(rte, pred, average="macro"),
            "weighted_f1": f1_score(rte, pred, average="weighted"),
        })

    return pd.DataFrame(rows)

# -------------------------
# 2-way hierarchical CV (soft/hard routing)
# -------------------------
def hierarchical_cv(stage1, stage2_upper, stage2_lower, X, y, region2, skf, all_classes, routing="soft"):
    """
    routing:
      - "soft": mixture-of-experts: P(class)=P(Upper)*P(class|Upper)+P(Lower)*P(class|Lower)
      - "hard": choose region by argmax stage1, then use only that expert
    """
    rows = []
    all_classes = list(all_classes)
    cls_to_col = {c: i for i, c in enumerate(all_classes)}

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region2.iloc[tr], region2.iloc[te]

        s1 = clone(stage1)
        u  = clone(stage2_upper)
        l  = clone(stage2_lower)

        # Stage 1
        s1.fit(Xtr, rtr)
        s1_proba = s1.predict_proba(Xte)

        s1_classes = list(s1.steps[-1][1].classes_) if isinstance(s1, Pipeline) else list(s1.classes_)
        idx_upper = s1_classes.index("Upper")
        idx_lower = s1_classes.index("Lower")
        p_upper = s1_proba[:, idx_upper]
        p_lower = s1_proba[:, idx_lower]

        # Stage 2 experts trained on TRUE region within train fold
        u.fit(Xtr.loc[rtr == "Upper"], ytr.loc[rtr == "Upper"])
        l.fit(Xtr.loc[rtr == "Lower"], ytr.loc[rtr == "Lower"])

        pu_raw = u.predict_proba(Xte)
        pl_raw = l.predict_proba(Xte)

        u_classes = list(u.steps[-1][1].classes_) if isinstance(u, Pipeline) else list(u.classes_)
        l_classes = list(l.steps[-1][1].classes_) if isinstance(l, Pipeline) else list(l.classes_)

        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)

        if routing == "soft":
            for j, c in enumerate(u_classes):
                full_proba[:, cls_to_col[c]] += p_upper * pu_raw[:, j]
            for j, c in enumerate(l_classes):
                full_proba[:, cls_to_col[c]] += p_lower * pl_raw[:, j]

        elif routing == "hard":
            pick_upper = (p_upper >= p_lower)
            for j, c in enumerate(u_classes):
                full_proba[pick_upper, cls_to_col[c]] = pu_raw[pick_upper, j]
            for j, c in enumerate(l_classes):
                full_proba[~pick_upper, cls_to_col[c]] = pl_raw[~pick_upper, j]
        else:
            raise ValueError("routing must be 'soft' or 'hard'")

        # Renormalize
        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]
        metrics = summarize_metrics(yte, ypred, full_proba, all_classes)
        metrics["fold"] = fold
        rows.append(metrics)

    return pd.DataFrame(rows)

# -------------------------
# 3-way hierarchical CV (soft mixture)
# -------------------------
def hierarchical3_cv(stage1, expert_models, X, y, region3, skf, all_classes):
    """
    3-way mixture of experts:
      regions: Upper / Trunk / Lower
      P(class) = sum_r P(r) * P(class | r)
    expert_models: dict like {"Upper": model, "Trunk": model, "Lower": model}
    """
    rows = []
    all_classes = list(all_classes)
    cls_to_col = {c: i for i, c in enumerate(all_classes)}
    regions = ["Upper", "Trunk", "Lower"]

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region3.iloc[tr], region3.iloc[te]

        s1 = clone(stage1)
        s1.fit(Xtr, rtr)
        s1_proba = s1.predict_proba(Xte)
        s1_classes = list(s1.steps[-1][1].classes_) if isinstance(s1, Pipeline) else list(s1.classes_)

        # train experts on TRUE region
        trained = {}
        for reg in regions:
            m = clone(expert_models[reg])
            m.fit(Xtr.loc[rtr == reg], ytr.loc[rtr == reg])
            trained[reg] = m

        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)

        for reg in regions:
            p_reg = s1_proba[:, s1_classes.index(reg)]
            m = trained[reg]
            proba_raw = m.predict_proba(Xte)
            m_classes = list(m.steps[-1][1].classes_) if isinstance(m, Pipeline) else list(m.classes_)
            for j, c in enumerate(m_classes):
                full_proba[:, cls_to_col[c]] += p_reg * proba_raw[:, j]

        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]
        metrics = summarize_metrics(yte, ypred, full_proba, all_classes)
        metrics["fold"] = fold
        rows.append(metrics)

    return pd.DataFrame(rows)

# -------------------------
# Choose splits
# -------------------------
min_class_count = int(y.value_counts().min())
n_splits = min(5, min_class_count)
if n_splits < 2:
    raise ValueError(f"Not enough samples for CV. Min class count = {min_class_count}")
print("Min class count:", min_class_count)
print("Using n_splits:", n_splits)

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
all_classes = sorted(y.unique())

# -------------------------
# Baseline models (multiclass)
# -------------------------
baseline_models = {
    "logreg_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),

    "rf_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=800,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),

    "lightgbm_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),

    "xgb_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=600,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),

    "catboost_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=800,
            learning_rate=0.05,
            depth=6,
            loss_function="MultiClass",
            random_seed=RANDOM_SEED,
            verbose=False
        ))
    ]),
}

# --- Baseline CV ---
baseline_results = []
baseline_details = {}

for name, model in baseline_models.items():
    df = evaluate_model_cv(model, X, y, skf, all_classes)
    baseline_details[name] = df
    mean_row = df.mean(numeric_only=True).to_dict()
    mean_row["model"] = name
    baseline_results.append(mean_row)

baseline_summary = (
    pd.DataFrame(baseline_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(baseline_summary)

# -------------------------
# Stage 1 models (2-way: Upper/Lower)
# -------------------------
stage1_models = {
    "s1_logreg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),

    "s1_lightgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=350,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),

    "s1_xgb": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),

    "s1_catboost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=600,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            random_seed=RANDOM_SEED,
            verbose=False
        ))
    ]),
}

# --- Stage 1 CV summary ---
stage1_summary_rows = []
for name, model in stage1_models.items():
    df = evaluate_stage1_cv(model, X, region, skf)
    row = df.mean(numeric_only=True).to_dict()
    row["model"] = name
    stage1_summary_rows.append(row)

stage1_summary = pd.DataFrame(stage1_summary_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(stage1_summary)

# -------------------------
# Stage 2 expert models (Upper and Lower)
# -------------------------
stage2_upper_models = {
    "upper_lgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=600,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "upper_xgb": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=700,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
    "upper_catboost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=900,
            learning_rate=0.05,
            depth=6,
            loss_function="MultiClass",
            random_seed=RANDOM_SEED,
            verbose=False
        ))
    ]),
}

stage2_lower_models = {
    "lower_lgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=900,
            learning_rate=0.03,
            num_leaves=63,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "lower_xgb": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=900,
            learning_rate=0.04,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
    "lower_catboost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=1200,
            learning_rate=0.04,
            depth=7,
            loss_function="MultiClass",
            random_seed=RANDOM_SEED,
            verbose=False
        ))
    ]),
}

# -------------------------
# 2-way hierarchical CV (SOFT) across combinations
# -------------------------
hier_results = []
hier_details = {}

for s1_name, s1 in stage1_models.items():
    for u_name, u in stage2_upper_models.items():
        for l_name, l in stage2_lower_models.items():
            key = f"{s1_name}__{u_name}__{l_name}"
            df = hierarchical_cv(s1, u, l, X, y, region, skf, all_classes, routing="soft")
            hier_details[key] = df
            mean_row = df.mean(numeric_only=True).to_dict()
            mean_row["model"] = key
            hier_results.append(mean_row)

hier_summary = (
    pd.DataFrame(hier_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
display(hier_summary)

# -------------------------
# Soft vs hard routing using best hierarchical combo
# -------------------------
best_hier_key = hier_summary.iloc[0]["model"]
best_parts = best_hier_key.split("__")
best_s1 = stage1_models[best_parts[0]]
best_u  = stage2_upper_models[best_parts[1]]
best_l  = stage2_lower_models[best_parts[2]]

soft_df = hierarchical_cv(best_s1, best_u, best_l, X, y, region, skf, all_classes, routing="soft")
hard_df = hierarchical_cv(best_s1, best_u, best_l, X, y, region, skf, all_classes, routing="hard")

print("SOFT routing mean:")
display(soft_df.mean(numeric_only=True))

print("HARD routing mean:")
display(hard_df.mean(numeric_only=True))

# -------------------------
# 3-way hierarchical CV (soft)
# -------------------------
stage1_3way = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1
    ))
])

experts_3way = {
    "Upper": stage2_upper_models["upper_lgbm"],
    "Trunk": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=350,
            learning_rate=0.05,
            num_leaves=15,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "Lower": stage2_lower_models["lower_lgbm"],
}

hier3_df = hierarchical3_cv(stage1_3way, experts_3way, X, y, region3, skf, all_classes)

print("3-way hierarchical mean:")
display(hier3_df.mean(numeric_only=True))
display(hier3_df)

# -------------------------
# Best baseline vs best hierarchical comparison
# -------------------------
best_base = baseline_summary.iloc[0]["model"]
base_mean = baseline_details[best_base].mean(numeric_only=True)
hier_mean = hier_details[best_hier_key].mean(numeric_only=True)

compare = pd.DataFrame({
    "baseline_best_mean": base_mean,
    "hierarchical_best_mean": hier_mean
})
compare["delta(hier-base)"] = compare["hierarchical_best_mean"] - compare["baseline_best_mean"]
display(compare)

# -------------------------
# Per-class report (simple holdout)
# -------------------------
from sklearn.metrics import classification_report

Xtr, Xte, ytr, yte, rtr, rte = train_test_split(
    X, y, region, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# Baseline best
base_model = clone(baseline_models[best_base])
base_model.fit(Xtr, ytr)
base_pred = base_model.predict(Xte)

print("BASELINE (best):", best_base)
print(classification_report(yte, base_pred, zero_division=0))

# Hierarchical best (soft mixture)
s1 = clone(best_s1)
u  = clone(best_u)
l  = clone(best_l)

s1.fit(Xtr, rtr)
p = s1.predict_proba(Xte)
s1_classes = list(s1.steps[-1][1].classes_) if isinstance(s1, Pipeline) else list(s1.classes_)
p_upper = p[:, s1_classes.index("Upper")]
p_lower = p[:, s1_classes.index("Lower")]

u.fit(Xtr[rtr == "Upper"], ytr[rtr == "Upper"])
l.fit(Xtr[rtr == "Lower"], ytr[rtr == "Lower"])

pu = u.predict_proba(Xte)
pl = l.predict_proba(Xte)

u_classes = list(u.steps[-1][1].classes_) if isinstance(u, Pipeline) else list(u.classes_)
l_classes = list(l.steps[-1][1].classes_) if isinstance(l, Pipeline) else list(l.classes_)

full = np.zeros((len(Xte), len(all_classes)))
col = {c: i for i, c in enumerate(all_classes)}

for j, c in enumerate(u_classes):
    full[:, col[c]] += p_upper * pu[:, j]
for j, c in enumerate(l_classes):
    full[:, col[c]] += p_lower * pl[:, j]

full = full / np.maximum(full.sum(axis=1, keepdims=True), 1e-12)
hier_pred = np.array(all_classes)[full.argmax(axis=1)]

print("HIERARCHICAL (best soft):", best_hier_key)
print(classification_report(yte, hier_pred, zero_division=0))


Min class count: 46
Using n_splits: 5


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3 4 5 6 7 8 9], got ['ExcessiveForwardLean' 'ForwardHead' 'LeftArmFallForward'
 'LeftAsymmetricalWeightShift' 'LeftKneeMovesOutward'
 'LeftShoulderElevation' 'RightArmFallForward' 'RightKneeMovesInward'
 'RightKneeMovesOutward' 'RightShoulderElevation']

## Hierarchical CV evaluation (stage 1 -> stage 2)

In [165]:
def hierarchical_cv(stage1, stage2_upper, stage2_lower, X, y, region, skf, all_classes):
    rows = []
    all_classes = list(all_classes)

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region.iloc[tr], region.iloc[te]

        # =========================
        # Stage 1: Region (Upper/Lower)
        # =========================
        stage1.fit(Xtr, rtr)
        s1_proba = stage1.predict_proba(Xte)

        if isinstance(stage1, Pipeline):
            s1_classes = list(stage1.steps[-1][1].classes_)
        else:
            s1_classes = list(stage1.classes_)

        idx_upper = s1_classes.index("Upper")
        idx_lower = s1_classes.index("Lower")

        p_upper = s1_proba[:, idx_upper]
        p_lower = s1_proba[:, idx_lower]

        # =========================
        # Stage 2: Specialized models (trained on TRUE region within train fold)
        # =========================
        Xtr_upper = Xtr.loc[rtr == "Upper"]
        ytr_upper = ytr.loc[rtr == "Upper"]
        Xtr_lower = Xtr.loc[rtr == "Lower"]
        ytr_lower = ytr.loc[rtr == "Lower"]

        stage2_upper.fit(Xtr_upper, ytr_upper)
        stage2_lower.fit(Xtr_lower, ytr_lower)

        # Conditional probs for ALL test rows (safe: model will only output its known classes)
        pu_raw = stage2_upper.predict_proba(Xte)
        pl_raw = stage2_lower.predict_proba(Xte)

        # Get which classes each stage2 model actually predicts
        if isinstance(stage2_upper, Pipeline):
            upper_classes = list(stage2_upper.steps[-1][1].classes_)
            lower_classes = list(stage2_lower.steps[-1][1].classes_)
        else:
            upper_classes = list(stage2_upper.classes_)
            lower_classes = list(stage2_lower.classes_)

        # =========================
        # Combine into full-class probability matrix
        # P(class) = P(region) * P(class | region)
        # Only fill the columns that exist for each stage2 model.
        # =========================
        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)
        cls_to_col = {c: i for i, c in enumerate(all_classes)}

        for j, c in enumerate(upper_classes):
            full_proba[:, cls_to_col[c]] = p_upper * pu_raw[:, j]

        for j, c in enumerate(lower_classes):
            full_proba[:, cls_to_col[c]] = p_lower * pl_raw[:, j]

        # Renormalize to keep probabilities valid + log_loss stable
        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        # Predict final class
        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]

        # Score
        m = summarize_metrics(yte, ypred, full_proba, all_classes)
        m["fold"] = fold
        rows.append(m)

    return pd.DataFrame(rows)

all_classes = sorted(y.unique())

hier_results = []
hier_details = {}

for s1_name, s1 in stage1_models.items():
    for s2_name, s2 in stage2_models.items():
        key = f"{s1_name}__{s2_name}"
        df = hierarchical_cv(
            stage1=clone(s1),
            stage2_upper=clone(s2),
            stage2_lower=clone(s2),
            X=X, y=y, region=region,
            skf=skf,
            all_classes=all_classes
        )

        hier_details[key] = df
        mean_row = df.mean(numeric_only=True).to_dict()
        mean_row["model"] = key
        hier_results.append(mean_row)

hier_summary = (
    pd.DataFrame(hier_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(hier_summary)


,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,r2_onehot_vs_proba,fold,model
0,0.643261,0.507883,0.517042,0.630161,1.630610,0.274964,3.0,s1_lightgbm__s2_lightgbm
1,0.592213,0.482674,0.456971,0.582840,1.663129,0.184237,3.0,s1_logreg__s2_lightgbm


## Soft vs hard routing

In [166]:
def hierarchical_cv(stage1, stage2_upper, stage2_lower, X, y, region, skf, all_classes, routing="soft"):
    """
    routing:
      - "soft": mixture of experts (your current method)
      - "hard": pick region by argmax from stage1, then use ONLY that branch
    """
    rows = []
    all_classes = list(all_classes)
    cls_to_col = {c: i for i, c in enumerate(all_classes)}

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region.iloc[tr], region.iloc[te]

        # Stage 1
        stage1.fit(Xtr, rtr)
        s1_proba = stage1.predict_proba(Xte)

        if isinstance(stage1, Pipeline):
            s1_classes = list(stage1.steps[-1][1].classes_)
        else:
            s1_classes = list(stage1.classes_)

        idx_upper = s1_classes.index("Upper")
        idx_lower = s1_classes.index("Lower")
        p_upper = s1_proba[:, idx_upper]
        p_lower = s1_proba[:, idx_lower]

        # Stage 2 train on TRUE region
        Xtr_upper = Xtr.loc[rtr == "Upper"]
        ytr_upper = ytr.loc[rtr == "Upper"]
        Xtr_lower = Xtr.loc[rtr == "Lower"]
        ytr_lower = ytr.loc[rtr == "Lower"]

        stage2_upper.fit(Xtr_upper, ytr_upper)
        stage2_lower.fit(Xtr_lower, ytr_lower)

        pu_raw = stage2_upper.predict_proba(Xte)
        pl_raw = stage2_lower.predict_proba(Xte)

        if isinstance(stage2_upper, Pipeline):
            upper_classes = list(stage2_upper.steps[-1][1].classes_)
            lower_classes = list(stage2_lower.steps[-1][1].classes_)
        else:
            upper_classes = list(stage2_upper.classes_)
            lower_classes = list(stage2_lower.classes_)

        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)

        if routing == "soft":
            # Soft mixture (current)
            for j, c in enumerate(upper_classes):
                full_proba[:, cls_to_col[c]] = p_upper * pu_raw[:, j]
            for j, c in enumerate(lower_classes):
                full_proba[:, cls_to_col[c]] = p_lower * pl_raw[:, j]

        elif routing == "hard":
            # Hard routing: pick region by argmax
            pick_upper = (p_upper >= p_lower)

            # Fill upper rows using upper model only
            for j, c in enumerate(upper_classes):
                full_proba[pick_upper, cls_to_col[c]] = pu_raw[pick_upper, j]

            # Fill lower rows using lower model only
            for j, c in enumerate(lower_classes):
                full_proba[~pick_upper, cls_to_col[c]] = pl_raw[~pick_upper, j]

        else:
            raise ValueError("routing must be 'soft' or 'hard'")

        # Renormalize
        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]
        m = summarize_metrics(yte, ypred, full_proba, all_classes)
        m["fold"] = fold
        rows.append(m)

    return pd.DataFrame(rows)

# --- Compare routing strategies with your best models ---
all_classes = sorted(y.unique())

s1_best = stage1_models["s1_lightgbm"]
u_best  = stage2_upper_models["upper_lgbm"]
l_best  = stage2_lower_models["lower_lgbm"]

soft_df = hierarchical_cv(clone(s1_best), clone(u_best), clone(l_best), X, y, region, skf, all_classes, routing="soft")
hard_df = hierarchical_cv(clone(s1_best), clone(u_best), clone(l_best), X, y, region, skf, all_classes, routing="hard")

print("SOFT routing mean:")
display(soft_df.mean(numeric_only=True))

print("HARD routing mean:")
display(hard_df.mean(numeric_only=True))


SOFT routing mean:


accuracy              0.642683
balanced_accuracy     0.506574
macro_f1              0.517034
weighted_f1           0.630608
log_loss              1.666366
r2_onehot_vs_proba    0.270090
fold                  3.000000
dtype: float64

HARD routing mean:


accuracy              0.639783
balanced_accuracy     0.498834
macro_f1              0.508806
weighted_f1           0.627145
log_loss              5.678704
r2_onehot_vs_proba    0.190328
fold                  3.000000
dtype: float64

## 3 way hierarchical CV (Soft)

In [167]:
def hierarchical3_cv(stage1, expert_models, X, y, region3, skf, all_classes):
    """
    3-way mixture of experts:
      regions: Upper / Trunk / Lower
      P(class) = sum_r P(r) * P(class | r)
    expert_models: dict like {"Upper": model, "Trunk": model, "Lower": model}
    """
    rows = []
    all_classes = list(all_classes)
    cls_to_col = {c: i for i, c in enumerate(all_classes)}
    regions = ["Upper", "Trunk", "Lower"]

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region3.iloc[tr], region3.iloc[te]

        # --- Stage 1: 3-class region ---
        s1 = clone(stage1)
        s1.fit(Xtr, rtr)
        s1_proba = s1.predict_proba(Xte)

        if isinstance(s1, Pipeline):
            s1_classes = list(s1.steps[-1][1].classes_)
        else:
            s1_classes = list(s1.classes_)

        # --- Train experts on true region ---
        trained = {}
        for reg in regions:
            m = clone(expert_models[reg])
            m.fit(Xtr.loc[rtr == reg], ytr.loc[rtr == reg])
            trained[reg] = m

        # --- Combine probabilities ---
        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)

        for reg in regions:
            p_reg = s1_proba[:, s1_classes.index(reg)]

            m = trained[reg]
            proba_raw = m.predict_proba(Xte)

            if isinstance(m, Pipeline):
                m_classes = list(m.steps[-1][1].classes_)
            else:
                m_classes = list(m.classes_)

            for j, c in enumerate(m_classes):
                full_proba[:, cls_to_col[c]] += p_reg * proba_raw[:, j]

        # Renormalize
        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]
        m = summarize_metrics(yte, ypred, full_proba, all_classes)
        m["fold"] = fold
        rows.append(m)

    return pd.DataFrame(rows)


In [168]:
# --- Stage 1 (3-class): Upper / Trunk / Lower ---
stage1_3way = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1
    ))
])

# --- Experts for each region ---
experts_3way = {
    "Upper": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "Trunk": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "Lower": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
}

all_classes = sorted(y.unique())

hier3_df = hierarchical3_cv(
    stage1=stage1_3way,
    expert_models=experts_3way,
    X=X, y=y, region3=region3,
    skf=skf,
    all_classes=all_classes
)

print("3-way hierarchical mean:")
display(hier3_df.mean(numeric_only=True))
display(hier3_df)


3-way hierarchical mean:


accuracy              0.640945
balanced_accuracy     0.503482
macro_f1              0.514327
weighted_f1           0.628140
log_loss              1.859853
r2_onehot_vs_proba    0.250488
fold                  3.000000
dtype: float64

,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,r2_onehot_vs_proba,fold
0,0.617391,0.472484,0.473045,0.600754,2.104475,0.178671,1
1,0.689855,0.555207,0.558531,0.678005,1.584789,0.327071,2
2,0.634783,0.504743,0.517805,0.624122,1.870144,0.257944,3
3,0.631884,0.482498,0.502399,0.617419,1.830682,0.231739,4
4,0.630814,0.502477,0.519853,0.620399,1.909176,0.257017,5


In [169]:
best_base = baseline_summary.iloc[0]["model"]
best_hier = hier_summary.iloc[0]["model"]

print("Best baseline:", best_base)
print("Best hierarchical:", best_hier)

base_mean = baseline_details[best_base].mean(numeric_only=True)
hier_mean = hier_details[best_hier].mean(numeric_only=True)

compare = pd.DataFrame({
    "baseline_best_mean": base_mean,
    "hierarchical_best_mean": hier_mean
})
compare["delta(hier-base)"] = compare["hierarchical_best_mean"] - compare["baseline_best_mean"]
display(compare)


Best baseline: lightgbm_balanced
Best hierarchical: s1_lightgbm__s2_lightgbm


,baseline_best_mean,hierarchical_best_mean,delta(hier-base)
accuracy,0.658342,0.643261,-0.015081
balanced_accuracy,0.542838,0.507883,-0.034955
macro_f1,0.555771,0.517042,-0.038728
weighted_f1,0.650334,0.630161,-0.020174
log_loss,1.623420,1.630610,0.007190
r2_onehot_vs_proba,0.269642,0.274964,0.005322
fold,3.000000,3.000000,0.000000


## Best model comparison 

In [170]:

def oof_predictions_baseline(best_model, X, y, skf, all_classes):
    oof_pred = np.empty(len(y), dtype=object)
    for tr, te in skf.split(X, y):
        m = clone(best_model)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof_pred[te] = m.predict(X.iloc[te])
    return oof_pred

def oof_predictions_hier(best_s1, best_u, best_l, X, y, region, skf, all_classes):
    oof_pred = np.empty(len(y), dtype=object)
    for tr, te in skf.split(X, y):
        df_fold = hierarchical_cv(
            stage1=clone(best_s1),
            stage2_upper=clone(best_u),
            stage2_lower=clone(best_l),
            X=X.iloc[tr].append(X.iloc[te]),   # hack avoided below
            y=y.iloc[tr].append(y.iloc[te]),
            region=region.iloc[tr].append(region.iloc[te]),
            skf=StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_SEED), # not ideal
            all_classes=all_classes
        )
    return oof_pred


## Per class

In [171]:


Xtr, Xte, ytr, yte, rtr, rte = train_test_split(
    X, y, region, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# Baseline best = lightgbm_balanced
base = clone(baseline_models["lightgbm_balanced"])
base.fit(Xtr, ytr)
base_pred = base.predict(Xte)

print("BASELINE")
print(classification_report(yte, base_pred, zero_division=0))

# Hierarchical best (use your best key)
s1 = clone(stage1_models["s1_lightgbm"])
u  = clone(stage2_models["s2_lightgbm"])
l  = clone(stage2_models["s2_lightgbm"])

# Fit stage1
s1.fit(Xtr, rtr)
p = s1.predict_proba(Xte)
s1_classes = list(s1.steps[-1][1].classes_)
p_upper = p[:, s1_classes.index("Upper")]
p_lower = p[:, s1_classes.index("Lower")]

# Fit stage2 on region-split train
u.fit(Xtr[rtr=="Upper"], ytr[rtr=="Upper"])
l.fit(Xtr[rtr=="Lower"], ytr[rtr=="Lower"])

pu = u.predict_proba(Xte); upper_classes = list(u.steps[-1][1].classes_)
pl = l.predict_proba(Xte); lower_classes = list(l.steps[-1][1].classes_)

all_classes = sorted(y.unique())
full = np.zeros((len(Xte), len(all_classes)))
col = {c:i for i,c in enumerate(all_classes)}
for j,c in enumerate(upper_classes): full[:, col[c]] = p_upper * pu[:, j]
for j,c in enumerate(lower_classes): full[:, col[c]] = p_lower * pl[:, j]
full = full / np.maximum(full.sum(axis=1, keepdims=True), 1e-12)

hier_pred = np.array(all_classes)[full.argmax(axis=1)]

print("HIERARCHICAL")
print(classification_report(yte, hier_pred, zero_division=0))


BASELINE
                             precision    recall  f1-score   support

       ExcessiveForwardLean       0.58      0.70      0.64        20
                ForwardHead       0.62      0.68      0.65        19
         LeftArmFallForward       0.58      0.58      0.58        66
LeftAsymmetricalWeightShift       0.59      0.71      0.65        14
       LeftKneeMovesOutward       0.75      0.60      0.67        10
      LeftShoulderElevation       0.17      0.10      0.12        10
        RightArmFallForward       0.65      0.73      0.69        91
       RightKneeMovesInward       0.33      0.11      0.17         9
      RightKneeMovesOutward       1.00      0.95      0.97        56
     RightShoulderElevation       0.70      0.64      0.67        50

                   accuracy                           0.68       345
                  macro avg       0.60      0.58      0.58       345
               weighted avg       0.67      0.68      0.67       345

HIERARCHICAL
         